# Validate pipeline

Production-grade validation for the 25-year master feature table. Run all 13 sections before modeling. See [docs/validation_checks.md](../docs/validation_checks.md) for what each check does and why it matters.

In [ ]:
import sys
from pathlib import Path
from datetime import datetime

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

import config
import pandas as pd

# Capture all print output for the validation report
_original_stdout = sys.stdout

class _TeeOutput:
    def __init__(self):
        self.buffer = []
    def write(self, s):
        _original_stdout.write(s)
        self.buffer.append(s)
    def flush(self):
        _original_stdout.flush()
    def getvalue(self):
        return "".join(self.buffer)

_tee = _TeeOutput()
sys.stdout = _tee

master_path = config.MASTER_FEATURES_PATH
if master_path.exists():
    df = pd.read_parquet(master_path)
    print("Master shape:", df.shape)
else:
    print("Master not found; run pipeline first.")
    df = None

In [ ]:
# --- 1. PIT Integrity (Most Critical) ---
fp_path = config.FUNDAMENTAL_PIT_PATH
if fp_path.exists():
    fp = pd.read_parquet(fp_path)
    fp["date"] = pd.to_datetime(fp["date"])
    datekey_col = "datekey" if "datekey" in fp.columns else ("art_datekey" if "art_datekey" in fp.columns else None)
    if datekey_col:
        fp[datekey_col] = pd.to_datetime(fp[datekey_col])
        null_datekey = fp[datekey_col].isna().mean()
        print(f"NULL {datekey_col} rate: {null_datekey:.1%}")
        # Expect ~5-10% for tickers with no filings yet at start of history; if > 30% ASOF join may be wrong
        violations = fp[fp[datekey_col].notna() & (fp[datekey_col] > fp["date"])]
        assert len(violations) == 0, "CRITICAL: lookahead in fundamental_pit"
        print("PIT datekey check: PASS (no datekey > date)")
    # 1b. Quality metrics change only on filing dates (sample AAPL)
    if df is not None and "ncfo_r2_adjusted" in df.columns:
        aapl = df[df["ticker"] == "AAPL"].sort_values("date")
        if len(aapl) > 0:
            quality_changes = aapl["ncfo_r2_adjusted"].diff().abs()
            change_dates = aapl[quality_changes > 0]["date"]
            n_per_year = len(change_dates) / max(1, (aapl["date"].max() - aapl["date"].min()).days / 365)
            print(f"AAPL ncfo_r2_adjusted changes/year: ~{n_per_year:.1f} (expect ~4)")
    # 1c. Staleness distribution
    if df is not None and "days_since_filing" in df.columns:
        print("\ndays_since_filing distribution:")
        print(df["days_since_filing"].describe())
        stale = df[df["days_since_filing"] > 365]
        print(f"Rows with filing > 365 days stale: {len(stale)} ({len(stale)/len(df):.1%})")
else:
    print("fundamental_pit.parquet not found; skip PIT check.")

In [ ]:
# --- 2. Survivorship Bias (Second Most Critical) ---
from lib import validation

universe_path = config.DAILY_UNIVERSE_PATH
if universe_path.exists():
    u = pd.read_parquet(universe_path)
    u["date"] = pd.to_datetime(u["date"])
    for ticker, delist_date in [("LEH", "2008-09-15"), ("HTZ", "2020-05-22")]:
        if ticker in u["ticker"].values:
            ok = validation.check_delisted_sequence(u, ticker, pd.Timestamp(delist_date))
            print(f"Delisted {ticker} ({delist_date}):", "PASS" if ok else "FAIL")
            if ticker == "LEH":
                leh = u[u["ticker"] == "LEH"].sort_values("date")
                if not leh.empty and "fwd_delisted_30d" in leh.columns:
                    last_row = leh.iloc[-1]
                    print(f"  LEH last date: {last_row['date'].date()}, fwd_delisted_30d: {last_row['fwd_delisted_30d']}")
        else:
            print(f"Delisted {ticker}: ticker not in universe (skip).")
    # 2b. fwd_delisted_90d fraction from master
    if df is not None and "fwd_delisted_90d" in df.columns:
        pct = df["fwd_delisted_90d"].mean()
        print(f"\nfwd_delisted_90d True: {pct:.1%} (expect ~2-5%)")
    # 2c. Ticker reappearance (gaps)
    ticker_ranges = u.groupby("ticker")["date"].agg(["min", "max"])
    print(f"\nTicker date ranges: {len(ticker_ranges)} tickers")
else:
    print("Universe parquet not found; skip survivorship check.")

In [ ]:
# --- 3. Duplicate Rows (Critical) ---
if df is not None:
    dupes = df.duplicated(subset=["ticker", "date"]).sum()
    print(f"Duplicate (ticker, date) rows: {dupes}")
    assert dupes == 0, "Master features has duplicate rows — universe or join bug"
    print("Duplicate check: PASS")

In [ ]:
# --- 4. Distribution Sanity ---
if df is not None:
    for col, (lo, hi) in [
        ("pe_pit", (0, 500)),
        ("pb_pit", (0, 50)),
        ("pcf_pit", (0, 200)),
        ("evebitda_pit", (0, 100)),
    ]:
        if col in df.columns:
            valid = df[col].dropna()
            out = valid[(valid < lo) | (valid > hi)]
            n_valid = df[col].notna().sum()
            pct = len(out) / n_valid if n_valid > 0 else 0
            print(f"{col}: {len(out)} values outside [{lo}, {hi}] ({pct:.1%})")
    print("\nret_12m distribution:")
    if "ret_12m" in df.columns:
        print(df["ret_12m"].describe(percentiles=[0.01, 0.05, 0.25, 0.5, 0.75, 0.95, 0.99]))
        extreme = df[df["ret_12m"] > 5][["ticker", "date", "ret_12m"]].head(10)
        if len(extreme) > 0:
            print("\nExtreme ret_12m > 5 (sample):")
            print(extreme)
        actions_path = config.DATA_DIR / "ACTIONS.parquet"
        if actions_path.exists():
            actions = pd.read_parquet(actions_path)
            if "action" in actions.columns:
                splits = actions[actions["action"].astype(str).str.lower().str.strip() == "split"][["ticker", "date", "value"]]
                if not splits.empty and len(extreme) > 0:
                    ext = df[df["ret_12m"] > 5].copy()
                    merged = ext.merge(splits, on="ticker", suffixes=("", "_split"))
                    suspect = merged[abs(merged["ret_12m"] - merged["value"]) < 0.5]
                    if len(suspect) > 0:
                        print("\nWARNING: ret_12m close to split ratio — check closeadj adjustment:")
                        print(suspect[["ticker", "date", "ret_12m", "value"]].head())
    print("\nvol_20d distribution:")
    if "vol_20d" in df.columns:
        print(df["vol_20d"].describe(percentiles=[0.01, 0.05, 0.95, 0.99]))
        high_vol = (df["vol_20d"] > 5).sum()
        if high_vol > 0:
            print(f"  Rows with vol_20d > 5 (data error): {high_vol}")
    if "ncfo_r2_adjusted" in df.columns:
        print("\nncfo_r2_adjusted distribution:")
        print(df["ncfo_r2_adjusted"].describe())

In [ ]:
# --- 5. Sector Relative Sanity ---
sr_path = config.SECTOR_RELATIVE_PATH
if sr_path.exists():
    sr = pd.read_parquet(sr_path)
    for col in ["pe_vs_sector", "pb_vs_sector", "pcf_vs_sector"]:
        if col in sr.columns:
            m = sr[col].dropna().median()
            print(f"{col} median: {m:.3f}  (expect ~1.0)")
    for col in ["roic_vs_sector", "ret_3m_vs_sector"]:
        if col in sr.columns:
            m = sr[col].dropna().mean()
            print(f"{col} mean: {m:.4f}  (expect ~0.0)")
else:
    print("sector_relative.parquet not found; skip.")

In [ ]:
# --- 6. Macro Features Temporal Sanity ---
macro_path = config.MACRO_FEATURES_PATH
if macro_path.exists():
    macro = pd.read_parquet(macro_path)
    macro["date"] = pd.to_datetime(macro["date"])
    # Yield curve: expect negative in Aug-Oct 2019 (inversion)
    if "yield_curve" in macro.columns:
        yc_2019 = macro[macro["date"].between("2019-08-01", "2019-10-31")]["yield_curve"].dropna()
        if len(yc_2019) > 0:
            yc_mean = yc_2019.mean()
            if yc_mean < 0:
                print("Yield curve Aug-Oct 2019: inverted OK")
            else:
                print(f"WARNING: Yield curve Aug-Oct 2019 mean={yc_mean:.3f} (expect < 0)")
        else:
            print("Yield curve: no data in Aug-Oct 2019 (date range may exclude it)")
    # VIX: expect Mar 2020 >> 2019
    if "vix" in macro.columns:
        vix_mar = macro[macro["date"].between("2020-03-01", "2020-03-31")]["vix"].mean()
        vix_2019 = macro[macro["date"].between("2019-01-01", "2019-12-31")]["vix"].mean()
        print(f"VIX Mar 2020: {vix_mar:.1f} vs 2019: {vix_2019:.1f}")
        if pd.notna(vix_mar) and pd.notna(vix_2019) and vix_2019 != 0 and vix_mar > vix_2019 * 2:
            print("VIX spike OK")
        else:
            print("WARNING: VIX should spike in Mar 2020 (expect > 2x 2019)")
    if "spy_regime_ma" in macro.columns:
        spy_crash = macro[macro["date"].between("2020-03-15", "2020-04-30")]["spy_regime_ma"].dropna()
        if len(spy_crash) > 0:
            print(f"SPY regime Mar 2020: mean={spy_crash.mean():.2f} (expect 0.0)")
    # CPI: expect 2022 elevated vs 2019
    if "cpi_yoy" in macro.columns:
        cpi_2022 = macro[macro["date"].between("2022-01-01", "2022-12-31")]["cpi_yoy"].mean()
        cpi_2019 = macro[macro["date"].between("2019-01-01", "2019-12-31")]["cpi_yoy"].mean()
        print(f"CPI YoY 2022: {cpi_2022:.1f}% vs 2019: {cpi_2019:.1f}%")
        if pd.notna(cpi_2022) and pd.notna(cpi_2019) and cpi_2019 != 0 and cpi_2022 > cpi_2019 * 2:
            print("CPI 2022 elevated OK")
        else:
            print("WARNING: CPI 2022 should be elevated vs 2019 (expect > 2x)")
else:
    print("macro_features.parquet not found; skip.")

AssertionError: Yield curve should invert Aug-Oct 2019

In [ ]:
# --- 7. Insider Signal Sanity ---
ins_path = config.INSIDER_INSTITUTIONAL_PATH
ins_source = df if (df is not None and "insider_buy_count_90d" in df.columns) else None
if ins_source is None and ins_path.exists():
    ins_source = pd.read_parquet(ins_path)
if ins_source is not None and "insider_buy_count_90d" in ins_source.columns:
    buy_pct = (ins_source["insider_buy_count_90d"] > 0).mean()
    if buy_pct < 0.01:
        print("WARNING: <1% insider buy activity — SF2 securityadcode filter likely still wrong")
        print("Re-run 06_insider_institutional.py after removing securityadcode filter")
    else:
        print(f"Insider buy activity: {buy_pct:.1%}  OK")
    if df is not None and "scalemarketcap" in df.columns:
        merged = ins_source.merge(df[["ticker", "date", "scalemarketcap"]], on=["ticker", "date"], how="inner")
        if "scalemarketcap" in merged.columns:
            by_size = merged.groupby("scalemarketcap")["insider_buy_count_90d"].apply(lambda x: (x > 0).mean())
            print("Insider buy rate by scalemarketcap:")
            print(by_size.sort_index())
else:
    print("Insider data not found; skip.")

In [ ]:
# --- 8. Cross-Feature Consistency ---
if df is not None:
    if "pcf_pit" in df.columns and "pe_pit" in df.columns:
        corr = df[["pcf_pit", "pe_pit"]].corr().iloc[0, 1]
        print(f"pcf_pit vs pe_pit correlation: {corr:.3f} (expect 0.3-0.7)")
    daily_path = config.DATA_DIR / "DAILY.parquet"
    if daily_path.exists() and "pe_pit" in df.columns:
        daily = pd.read_parquet(daily_path)
        daily["date"] = pd.to_datetime(daily["date"])
        if "pe" in daily.columns:
            check = df[["ticker", "date", "pe_pit"]].merge(
                daily[["ticker", "date", "pe"]], on=["ticker", "date"], how="inner"
            ).dropna()
            if len(check) > 100:
                corr = check["pe_pit"].corr(check["pe"])
                ratio = (check["pe_pit"] / check["pe"].replace(0, float("nan"))).median()
                print(f"pe_pit vs DAILY.pe correlation: {corr:.3f} (expect >0.95)")
                print(f"pe_pit / DAILY.pe median ratio: {ratio:.3f} (expect ~1.0)")

In [ ]:
# --- 9. Temporal Consistency ---
if df is not None:
    aapl = df[df["ticker"] == "AAPL"].sort_values("date")
    if len(aapl) > 1:
        if "vol_20d" in aapl.columns:
            vol_ratio = aapl["vol_20d"].pct_change().abs()
            spikes = aapl[vol_ratio > 2.0][["date", "vol_20d"]]
            print(f"AAPL vol_20d spikes >200% overnight: {len(spikes)}")
            if len(spikes) > 0:
                print(spikes.head())
        if "pcf_pit" in aapl.columns and "days_since_filing" in aapl.columns:
            pcf_changes = aapl["pcf_pit"].pct_change().abs()
            big_jumps = aapl[pcf_changes > 0.5][["date", "pcf_pit", "days_since_filing"]]
            print(f"\nAAPL pcf_pit jumps >50% overnight: {len(big_jumps)}")
            if len(big_jumps) > 0:
                print(big_jumps.head())

In [ ]:
# --- 10. Null Rate Audit ---
if df is not None:
    null_rates = df.isnull().mean().sort_values(ascending=False)
    print(null_rates.head(30))
    high_null = null_rates[null_rates > 0.8]
    if len(high_null) > 0:
        print(f"\nFeatures with >80% null (investigate): {list(high_null.index)}")

In [ ]:
# --- 11. Null Rate by Year ---
if df is not None:
    df_temp = df.copy()
    df_temp["date"] = pd.to_datetime(df_temp["date"])
    df_temp["year"] = df_temp["date"].dt.year
    cols = [c for c in ["pcf_pit", "ncfo_r2_adjusted", "inst_shrunits", "yield_curve"] if c in df_temp.columns]
    if cols:
        null_by_year = df_temp.groupby("year")[cols].apply(lambda x: x.isna().mean())
        print(null_by_year)
        print("# inst_shrunits ~100% null before 2013; ncfo_r2_adjusted high null before 2003; yield_curve 0% null")

In [ ]:
# --- 12. Universe Composition Over Time ---
if config.DAILY_UNIVERSE_PATH.exists():
    u = pd.read_parquet(config.DAILY_UNIVERSE_PATH)
    u["date"] = pd.to_datetime(u["date"])
    u["year"] = u["date"].dt.year
    in_col = "in_universe" if "in_universe" in u.columns else None
    if in_col:
        annual_counts = u[u[in_col]].groupby("year")["ticker"].nunique()
    else:
        annual_counts = u.groupby("year")["ticker"].nunique()
    print(annual_counts)
    print("# Should grow from ~3000 in 2000 to ~5000+ in 2020s; big drops = bug; flat = wrong filter")

In [ ]:
# --- 13. Restatement Coverage (Raw SF1) ---
def check_restatement_coverage(sf1: pd.DataFrame, ticker: str) -> pd.DataFrame:
    """For a given ticker show all ARQ rows grouped by reportperiod. Multiple rows = restatement history."""
    arq = sf1[sf1["ticker"] == ticker].sort_values(["reportperiod", "datekey"])
    if arq.empty:
        print(f"{ticker}: no ARQ data")
        return pd.DataFrame()
    counts = arq.groupby("reportperiod").size().reset_index(name="n_versions")
    multi = counts[counts.n_versions > 1]
    if len(multi) > 0:
        print(f"{ticker}: {len(multi)} periods with multiple filings")
        for period in multi["reportperiod"]:
            versions = arq[arq["reportperiod"] == period][["datekey", "reportperiod", "ncfo", "revenue", "netinccmn"]]
            print(versions.to_string())
    else:
        print(f"{ticker}: single filing per period, no amendments in data")
    return counts

sf1_path = config.DATA_DIR / "SF1.parquet"
sf1_alt = config.DATA_DIR / "sf1.parquet"
if sf1_path.exists() or sf1_alt.exists():
    import duckdb
    p = sf1_path if sf1_path.exists() else sf1_alt
    con = duckdb.connect()
    path_sql = repr(str(p.resolve()))
    sf1_df = con.execute(f"""
        SELECT ticker, dimension, datekey, reportperiod, ncfo, revenue, netinccmn
        FROM read_parquet({path_sql})
        WHERE ticker IN ('GE', 'UAA', 'NFLX', 'TSLA') AND dimension = 'ARQ'
    """).df()
    con.close()
    for ticker in ["GE", "UAA", "NFLX", "TSLA"]:
        check_restatement_coverage(sf1_df, ticker)
else:
    print("SF1.parquet not found; skip restatement check.")

In [ ]:
# --- Write validation report ---
sys.stdout = _original_stdout  # Restore stdout
report_content = _tee.getvalue()
report_path = config.OUTPUTS_DIR / "validation_report.md"
report_path.parent.mkdir(parents=True, exist_ok=True)
header = f"""# Validation Report

Generated: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}
Master: {config.MASTER_FEATURES_PATH}
Date range: {config.DATE_START} to {config.DATE_END}

---

"""
with open(report_path, "w", encoding="utf-8") as f:
    f.write(header)
    f.write("## Output\n\n```\n")
    f.write(report_content)
    f.write("\n```\n")
print(f"Report saved to {report_path}")